# STIR-Net V1 — first same-sample overfit

This notebook uses the **current hardened STIR-Net source implementation** and contains **no runtime monkey patches**.

Goal:

1. rebuild the same real all-cell BlastoSPIM sample used by the successful acceptance/backward gates;
2. measure a clean evaluation baseline;
3. train repeatedly on the **same complete biological scene**;
4. evaluate after each optimizer step;
5. verify that the optimization objective begins to decrease.

Default: **5 optimizer steps**. If the loss trends downward, extend to 10–20 later.


In [ ]:
from pathlib import Path
import gc
import json
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import torch

from learned.stirnet import RefinementCriterion, StirNet
from learned.stirnet.debugging.acceptance.first_overfit import (
    _reduced_config,
    _repo_root,
    build_real_batch,
)
from learned.stirnet.training.trainer import (
    model_forward_from_batch,
    move_to_device,
)

TRAIN_STEPS = 5
EVAL_EVERY = 1
SEED = 40266
AMP_DTYPE = torch.float16
GRAD_SCALER_INITIAL_SCALE = 1024.0

REPO_ROOT = _repo_root(Path.cwd())
DATA_DIR = (
    REPO_ROOT
    / "data"
    / "learned"
    / "stirnet"
    / "first_overfit"
    / "BlastoSPIM1_F22_030_034"
)
RUN_DIR = REPO_ROOT / "runs" / "stirnet" / "first_overfit" / "05_same_sample"
RUN_DIR.mkdir(parents=True, exist_ok=True)
HISTORY_PATH = RUN_DIR / "history.json"

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("This first-overfit experiment requires CUDA.")

device = torch.device("cuda")
properties = torch.cuda.get_device_properties(0)

print("Repository     :", REPO_ROOT)
print("Data           :", DATA_DIR)
print("Run directory  :", RUN_DIR)
print("GPU            :", properties.name)
print("Dedicated VRAM :", f"{properties.total_memory / 1024**3:.3f} GiB")
print("PyTorch        :", torch.__version__)
print("CUDA runtime   :", torch.version.cuda)
print("Train steps    :", TRAIN_STEPS)


## 1. Preflight

In [ ]:
required_files = [
    DATA_DIR / "raw_movie.npy",
    DATA_DIR / "instance_movie.npy",
    DATA_DIR / "markers_movie.npy",
    DATA_DIR / "gt_movie.npy",
    DATA_DIR / "metadata.json",
    DATA_DIR / "trackastra" / "track_graph.pkl",
    DATA_DIR / "stirnet_source" / "raw_norm_target.npy",
    DATA_DIR / "stirnet_source" / "foreground_target.npy",
    DATA_DIR / "stirnet_source" / "edt_target.npy",
    DATA_DIR / "stirnet_source" / "boundary_target.npy",
    DATA_DIR / "stirnet_source" / "marker_heatmap_target.npy",
    DATA_DIR / "stirnet_source" / "dref_um.npy",
]

missing = [path for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError(
        "The first-overfit preparation cache is incomplete. "
        "Run notebooks/stirnet/02_prepare_trackastra_first_overfit.ipynb first.\n\n"
        + "\n".join(str(path) for path in missing)
    )

print("Preparation cache is complete.")


## 2. Build the real all-cell sample

This calls the current source-level acceptance helper. It uses the same all-cell ROI, current instances, GT label map, and Trackastra temporal graph that passed the forward and one-step backward gates.


In [ ]:
prepare_started = time.perf_counter()
batch, sample = build_real_batch(DATA_DIR)

print("ROI shape          :", sample["roi_shape"])
print("Current cells      :", sample["current_count"])
print("GT cells           :", sample["target_count"])
print("Graph nodes        :", sample["graph_nodes"])
print("Temporal tracklets :", sample["temporal_tracklets"])
print("Required queries   :", sample["required_queries"])
print("Instance geom max  :", f"{sample['instance_geometry_max']:.5f}")
print("Graph geom max     :", f"{sample['graph_geometry_max']:.5f}")
print("CPU preparation    :", f"{time.perf_counter() - prepare_started:.2f} s")

assert sample["current_count"] == 36
assert sample["target_count"] == 33
assert sample["temporal_tracklets"] == 52
assert sample["required_queries"] == 132


## 3. Configure the reduced-width V1 training profile

In [ ]:
cfg = _reduced_config()

print("Spatial channels :", cfg.spatial.channels)
print("Temporal d_model :", cfg.temporal.d_model)
print("Coreason d_model :", cfg.coreasoning.d_model)
print("Query d_model    :", cfg.queries.d_model)
print("Decoder d_model  :", cfg.decoder.d_model)
print("Max queries      :", cfg.queries.max_queries)
print("Learning rate    :", cfg.training.lr)
print("Weight decay     :", cfg.training.weight_decay)
print("Max grad norm    :", cfg.training.max_grad_norm)
print(
    "Checkpointing    :",
    {
        "master": cfg.training.activation_checkpointing,
        "spatial": cfg.training.checkpoint_spatial,
        "coreasoning": cfg.training.checkpoint_coreasoning,
        "losses": cfg.training.checkpoint_losses,
    },
)

if not cfg.training.activation_checkpointing:
    raise RuntimeError("Activation checkpointing is expected for this experiment.")


## 4. Instantiate model, criterion, optimizer, and AMP scaler

In [ ]:
model = StirNet(cfg).to(device)
criterion = RefinementCriterion(
    cfg.losses,
    cfg.queries,
    cfg.training,
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.training.lr,
    weight_decay=cfg.training.weight_decay,
)

scaler = torch.amp.GradScaler(
    "cuda",
    init_scale=GRAD_SCALER_INITIAL_SCALE,
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("Trainable parameters:", f"{trainable_parameters:,}")
print("Initial AMP scale    :", scaler.get_scale())


## 5. Move model inputs to CUDA

Large target maps remain on CPU by design. The spatial volume is stored on CUDA as FP16, matching the validated gates.


In [ ]:
batch_device = {}

for key, value in batch.items():
    if key == "targets":
        batch_device[key] = value
    elif key == "spatial_inputs":
        batch_device[key] = value.to(device=device, dtype=AMP_DTYPE)
    elif key == "instance_labels":
        batch_device[key] = value.to(device=device, dtype=torch.int32)
    else:
        batch_device[key] = move_to_device(value, device)

del batch
gc.collect()
torch.cuda.empty_cache()

print(
    "spatial_inputs:",
    tuple(batch_device["spatial_inputs"].shape),
    batch_device["spatial_inputs"].dtype,
    batch_device["spatial_inputs"].device,
)
print(
    "target label map:",
    tuple(batch_device["targets"][0]["label_map"].shape),
    batch_device["targets"][0]["label_map"].dtype,
    batch_device["targets"][0]["label_map"].device,
)
print("CUDA allocated:", f"{torch.cuda.memory_allocated() / 1024**3:.3f} GiB")


## 6. Training/evaluation helpers

In [ ]:
GIB = 1024**3


def cuda_memory():
    return {
        "allocated_gib": torch.cuda.memory_allocated() / GIB,
        "reserved_gib": torch.cuda.memory_reserved() / GIB,
        "peak_gib": torch.cuda.max_memory_allocated() / GIB,
    }


def begin_phase():
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    return time.perf_counter()


def end_phase(started):
    torch.cuda.synchronize()
    return {
        "seconds": time.perf_counter() - started,
        **cuda_memory(),
    }


def finite_primary_outputs(outputs):
    tensors = {
        "exist_logits": outputs.exist_logits,
        "centers_cellscale": outputs.centers_cellscale,
        "coarse_mask_logits": outputs.coarse_mask_logits,
        "query_embeddings": outputs.query_embeddings,
    }
    for name, tensor in tensors.items():
        if not bool(torch.isfinite(tensor.float()).all()):
            raise RuntimeError(f"Non-finite model output: {name}")


def detached_loss_dict(losses):
    result = {}
    for name, value in losses.items():
        tensor = value.detach().float()
        if tensor.numel() != 1:
            continue
        scalar = float(tensor.cpu())
        if not math.isfinite(scalar):
            raise RuntimeError(f"Non-finite scalar loss: {name}={scalar}")
        result[name] = scalar
    return result


@torch.no_grad()
def evaluate():
    model.eval()
    criterion.eval()
    gc.collect()
    torch.cuda.empty_cache()

    started = begin_phase()
    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
        outputs = model_forward_from_batch(model, batch_device)
        losses = criterion(outputs, batch_device["targets"])
    metrics = end_phase(started)

    finite_primary_outputs(outputs)
    loss_values = detached_loss_dict(losses)

    del outputs, losses
    gc.collect()
    torch.cuda.empty_cache()

    return loss_values, metrics


def train_one_step(step):
    model.train()
    criterion.train()
    optimizer.zero_grad(set_to_none=True)

    gc.collect()
    torch.cuda.empty_cache()

    phase = "training forward"

    try:
        started = begin_phase()
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
            outputs = model_forward_from_batch(model, batch_device)
        forward_metrics = end_phase(started)
        finite_primary_outputs(outputs)

        phase = "matching/loss"
        started = begin_phase()
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
            losses = criterion(outputs, batch_device["targets"])
            loss = losses["loss"]
        loss_metrics = end_phase(started)

        if not bool(torch.isfinite(loss.float())):
            raise RuntimeError(
                f"Non-finite training loss at step {step}: {float(loss.detach())}"
            )

        loss_values = detached_loss_dict(losses)

        phase = "scaled backward"
        started = begin_phase()
        scaler.scale(loss).backward()
        backward_metrics = end_phase(started)

        phase = "gradient validation/clip"
        started = begin_phase()
        scaler.unscale_(optimizer)

        gradient_tensors = 0
        finite_gradient_tensors = 0
        nonzero_gradient_tensors = 0
        tracked_name = None
        tracked_parameter = None
        tracked_before = None

        for name, parameter in model.named_parameters():
            if parameter.grad is None:
                continue

            gradient_tensors += 1
            if not bool(torch.isfinite(parameter.grad.detach().float()).all()):
                raise RuntimeError(f"Non-finite gradient at step {step}: {name}")

            finite_gradient_tensors += 1

            if bool(torch.count_nonzero(parameter.grad)):
                nonzero_gradient_tensors += 1
                if tracked_parameter is None:
                    tracked_name = name
                    tracked_parameter = parameter
                    tracked_before = parameter.detach().clone()

        if gradient_tensors == 0:
            raise RuntimeError(f"No gradients were produced at step {step}.")
        if nonzero_gradient_tensors == 0:
            raise RuntimeError(f"All gradients are zero at step {step}.")

        preclip_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            cfg.training.max_grad_norm,
            error_if_nonfinite=True,
        )
        preclip_norm_value = float(preclip_norm.detach().float().cpu())

        if not math.isfinite(preclip_norm_value) or preclip_norm_value <= 0:
            raise RuntimeError(
                f"Invalid gradient norm at step {step}: {preclip_norm_value}"
            )

        gradient_metrics = end_phase(started)

        phase = "optimizer step"
        started = begin_phase()

        scale_before = float(scaler.get_scale())
        scaler.step(optimizer)
        scaler.update()
        optimizer_metrics = end_phase(started)
        scale_after = float(scaler.get_scale())

        tracked_delta = float(
            (tracked_parameter.detach() - tracked_before)
            .abs()
            .max()
            .float()
            .cpu()
        )

        if tracked_delta <= 0:
            raise RuntimeError(
                f"Optimizer did not change {tracked_name} at step {step}."
            )

        metrics = {
            "step": int(step),
            "train_loss": loss_values,
            "forward": forward_metrics,
            "loss_forward": loss_metrics,
            "backward": backward_metrics,
            "gradient_phase": gradient_metrics,
            "optimizer": optimizer_metrics,
            "gradient_tensors": int(gradient_tensors),
            "finite_gradient_tensors": int(finite_gradient_tensors),
            "nonzero_gradient_tensors": int(nonzero_gradient_tensors),
            "preclip_grad_norm": preclip_norm_value,
            "grad_clip_limit": float(cfg.training.max_grad_norm),
            "amp_scale_before": scale_before,
            "amp_scale_after": scale_after,
            "tracked_parameter": tracked_name,
            "tracked_parameter_max_delta": tracked_delta,
        }

        optimizer.zero_grad(set_to_none=True)
        del outputs, losses, loss, tracked_before

        gc.collect()
        torch.cuda.empty_cache()

        return metrics

    except torch.OutOfMemoryError:
        torch.cuda.synchronize()
        print(f"\nCUDA OOM during '{phase}' at optimizer step {step}")
        print(cuda_memory())
        raise


## 7. Step 0 — evaluation baseline

This is the clean reference against which post-update evaluation losses are compared.


In [ ]:
baseline_losses, baseline_metrics = evaluate()

print("Baseline evaluation")
print("-------------------")
for name, value in baseline_losses.items():
    print(f"{name:22s}: {value:.7f}")

print("\nEval time:", f"{baseline_metrics['seconds']:.2f} s")
print("Eval peak CUDA:", f"{baseline_metrics['peak_gib']:.3f} GiB")


## 8. Repeated same-sample training

Each successful step is immediately written to:

`runs/stirnet/first_overfit/05_same_sample/history.json`

so partial results survive a later interruption.


In [ ]:
history = [
    {
        "step": 0,
        "eval_loss": baseline_losses,
        "eval": baseline_metrics,
    }
]

with HISTORY_PATH.open("w", encoding="utf-8") as handle:
    json.dump(history, handle, indent=2)

experiment_started = time.perf_counter()

for step in range(1, TRAIN_STEPS + 1):
    print()
    print("=" * 72)
    print(f"OPTIMIZER STEP {step}/{TRAIN_STEPS}")
    print("=" * 72)

    step_started = time.perf_counter()
    train_metrics = train_one_step(step)

    print("train total loss      :", f"{train_metrics['train_loss']['loss']:.7f}")
    print("pre-clip grad norm    :", f"{train_metrics['preclip_grad_norm']:.7f}")
    print(
        "finite/nonzero grads  :",
        f"{train_metrics['finite_gradient_tensors']}/"
        f"{train_metrics['gradient_tensors']}, "
        f"nonzero={train_metrics['nonzero_gradient_tensors']}",
    )
    print(
        "AMP scale             :",
        f"{train_metrics['amp_scale_before']:.1f}"
        f" -> {train_metrics['amp_scale_after']:.1f}",
    )
    print(
        "tracked parameter Δ   :",
        f"{train_metrics['tracked_parameter_max_delta']:.9g}",
    )
    print(
        "model forward         :",
        f"{train_metrics['forward']['seconds']:.2f} s, "
        f"peak={train_metrics['forward']['peak_gib']:.3f} GiB",
    )
    print(
        "matching/loss         :",
        f"{train_metrics['loss_forward']['seconds']:.2f} s, "
        f"peak={train_metrics['loss_forward']['peak_gib']:.3f} GiB",
    )
    print(
        "backward              :",
        f"{train_metrics['backward']['seconds']:.2f} s, "
        f"peak={train_metrics['backward']['peak_gib']:.3f} GiB",
    )

    record = {**train_metrics}

    if step % EVAL_EVERY == 0:
        eval_losses, eval_metrics = evaluate()
        record["eval_loss"] = eval_losses
        record["eval"] = eval_metrics

        print("post-step eval loss   :", f"{eval_losses['loss']:.7f}")
        print(
            "post-step eval        :",
            f"{eval_metrics['seconds']:.2f} s, "
            f"peak={eval_metrics['peak_gib']:.3f} GiB",
        )

    record["wall_seconds"] = time.perf_counter() - step_started
    history.append(record)

    with HISTORY_PATH.open("w", encoding="utf-8") as handle:
        json.dump(history, handle, indent=2)

    print("step wall time         :", f"{record['wall_seconds']:.2f} s")

print()
print(
    "Training experiment time:",
    f"{time.perf_counter() - experiment_started:.2f} s",
)
print("History saved to:", HISTORY_PATH)


## 9. Summarize the loss trajectory

In [ ]:
eval_records = [record for record in history if "eval_loss" in record]

eval_steps = [record["step"] for record in eval_records]
eval_total = [record["eval_loss"]["loss"] for record in eval_records]

baseline = eval_total[0]
final = eval_total[-1]

absolute_change = final - baseline
relative_change = (final - baseline) / max(abs(baseline), 1e-12)

print("Evaluation loss trajectory")
print("--------------------------")

for step, loss_value in zip(eval_steps, eval_total):
    print(f"step {step:3d}: {loss_value:.7f}")

print()
print("Baseline eval loss :", f"{baseline:.7f}")
print("Final eval loss    :", f"{final:.7f}")
print("Absolute change    :", f"{absolute_change:+.7f}")
print("Relative change    :", f"{100 * relative_change:+.3f}%")

if final < baseline:
    print("\nPASS SIGNAL: evaluation loss is below the initial same-sample baseline.")
else:
    print(
        "\nNo decrease yet. Inspect the component losses before extending the run."
    )


## 10. Plot total training/evaluation loss

In [ ]:
train_records = [record for record in history if "train_loss" in record]

train_steps = [record["step"] for record in train_records]
train_total = [record["train_loss"]["loss"] for record in train_records]

plt.figure(figsize=(8, 5))

if train_steps:
    plt.plot(train_steps, train_total, marker="o", label="train-mode loss")

plt.plot(eval_steps, eval_total, marker="o", label="eval-mode loss")

plt.xlabel("Optimizer step")
plt.ylabel("Total loss")
plt.title("STIR-Net same-sample first overfit")
plt.grid(True, alpha=0.25)
plt.legend()
plt.show()


## 11. Inspect evaluation loss components

This table helps identify which terms move if the total loss stalls or oscillates.


In [ ]:
component_names = sorted(
    {
        name
        for record in eval_records
        for name in record["eval_loss"]
    }
)

print(
    f"{'component':24s}",
    *[f"s{step:>3d}" for step in eval_steps],
)

print("-" * (26 + 13 * len(eval_steps)))

for name in component_names:
    values = [
        record["eval_loss"].get(name, float("nan"))
        for record in eval_records
    ]

    print(
        f"{name:24s}",
        *[f"{value:12.6f}" for value in values],
    )


## Stop here

For this first run, the only question is:

> **Does the same-sample evaluation loss begin to decrease after a few real optimizer steps?**

If yes, extend `TRAIN_STEPS` to 10–20 in the next run.

Do not jump directly to long training until we inspect the total-loss trajectory, component losses, gradient norms, AMP scale, and backward memory/time.


In [ ]:
# ============================================================
# Continue the SAME in-memory overfit run for 20 more steps
# ============================================================

from learned.stirnet.training.checkpoint import save_checkpoint

ADDITIONAL_STEPS = 20
CHECKPOINT_EVERY = 5

# IMPORTANT:
# This assumes the notebook kernel has NOT been restarted.
# model, optimizer, scaler, batch_device, history, evaluate(),
# train_one_step(), cfg, RUN_DIR, and HISTORY_PATH must still exist.

required_objects = [
    "model",
    "optimizer",
    "scaler",
    "batch_device",
    "history",
    "evaluate",
    "train_one_step",
    "cfg",
    "RUN_DIR",
    "HISTORY_PATH",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Cannot continue the existing training state because these "
        f"notebook objects are missing: {missing_objects}. "
        "If the kernel was restarted, rerun Notebook 05 from the top."
    )


# ------------------------------------------------------------
# Determine where the previous run stopped
# ------------------------------------------------------------

completed_steps = [
    int(record["step"])
    for record in history
    if int(record["step"]) > 0
]

last_step = max(completed_steps, default=0)

start_step = last_step + 1
end_step = last_step + ADDITIONAL_STEPS

print("Continuing existing model state")
print("--------------------------------")
print("Previous final step :", last_step)
print("Next step           :", start_step)
print("Final planned step  :", end_step)
print("Additional updates  :", ADDITIONAL_STEPS)
print("Checkpoint interval :", CHECKPOINT_EVERY)


# ------------------------------------------------------------
# Save a checkpoint of the state BEFORE continuing
# ------------------------------------------------------------

pre_continue_checkpoint = (
    RUN_DIR
    / f"checkpoint_step_{last_step:03d}_before_continue.pt"
)

save_checkpoint(
    pre_continue_checkpoint,
    model=model,
    optimizer=optimizer,
    scaler=scaler,
    step=last_step,
    config=cfg,
    extra={
        "experiment": "same_sample_first_overfit",
        "history_path": str(HISTORY_PATH),
    },
)

print(
    "\nSaved starting checkpoint:",
    pre_continue_checkpoint,
)


# ------------------------------------------------------------
# Continue training
# ------------------------------------------------------------

continuation_started = time.perf_counter()

for step in range(start_step, end_step + 1):

    print()
    print("=" * 78)
    print(
        f"CONTINUED OPTIMIZER STEP "
        f"{step} / {end_step}"
    )
    print("=" * 78)

    step_started = time.perf_counter()

    # --------------------------------------------------------
    # One real training update
    # --------------------------------------------------------

    train_metrics = train_one_step(step)

    print(
        "train total loss      :",
        f"{train_metrics['train_loss']['loss']:.7f}",
    )

    print(
        "pre-clip grad norm    :",
        f"{train_metrics['preclip_grad_norm']:.7f}",
    )

    print(
        "finite/nonzero grads  :",
        f"{train_metrics['finite_gradient_tensors']}/"
        f"{train_metrics['gradient_tensors']}, "
        f"nonzero="
        f"{train_metrics['nonzero_gradient_tensors']}",
    )

    print(
        "AMP scale             :",
        f"{train_metrics['amp_scale_before']:.1f}"
        f" -> "
        f"{train_metrics['amp_scale_after']:.1f}",
    )

    print(
        "tracked parameter Δ   :",
        f"{train_metrics['tracked_parameter_max_delta']:.9g}",
    )

    print(
        "model forward         :",
        f"{train_metrics['forward']['seconds']:.2f} s, "
        f"peak="
        f"{train_metrics['forward']['peak_gib']:.3f} GiB",
    )

    print(
        "matching/loss         :",
        f"{train_metrics['loss_forward']['seconds']:.2f} s, "
        f"peak="
        f"{train_metrics['loss_forward']['peak_gib']:.3f} GiB",
    )

    print(
        "backward              :",
        f"{train_metrics['backward']['seconds']:.2f} s, "
        f"peak="
        f"{train_metrics['backward']['peak_gib']:.3f} GiB",
    )


    # --------------------------------------------------------
    # Deterministic post-step evaluation
    # --------------------------------------------------------

    eval_losses, eval_metrics = evaluate()

    print(
        "post-step eval loss   :",
        f"{eval_losses['loss']:.7f}",
    )

    print(
        "post-step eval        :",
        f"{eval_metrics['seconds']:.2f} s, "
        f"peak="
        f"{eval_metrics['peak_gib']:.3f} GiB",
    )


    # --------------------------------------------------------
    # Store history
    # --------------------------------------------------------

    record = {
        **train_metrics,
        "eval_loss": eval_losses,
        "eval": eval_metrics,
        "wall_seconds": (
            time.perf_counter()
            - step_started
        ),
    }

    history.append(record)


    # --------------------------------------------------------
    # Persist history after EVERY successful step
    # --------------------------------------------------------

    with HISTORY_PATH.open(
        "w",
        encoding="utf-8",
    ) as handle:

        json.dump(
            history,
            handle,
            indent=2,
        )


    # --------------------------------------------------------
    # Save resumable checkpoint every 5 steps
    # --------------------------------------------------------

    if (
        step % CHECKPOINT_EVERY == 0
        or step == end_step
    ):

        checkpoint_path = (
            RUN_DIR
            / f"checkpoint_step_{step:03d}.pt"
        )

        save_checkpoint(
            checkpoint_path,
            model=model,
            optimizer=optimizer,
            scaler=scaler,
            step=step,
            config=cfg,
            extra={
                "experiment": (
                    "same_sample_first_overfit"
                ),
                "history_path": str(
                    HISTORY_PATH
                ),
            },
        )

        print(
            "checkpoint saved     :",
            checkpoint_path,
        )


    print(
        "step wall time         :",
        f"{record['wall_seconds']:.2f} s",
    )


# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

continuation_seconds = (
    time.perf_counter()
    - continuation_started
)

eval_records = [
    record
    for record in history
    if "eval_loss" in record
]

eval_steps = [
    int(record["step"])
    for record in eval_records
]

eval_total = [
    float(record["eval_loss"]["loss"])
    for record in eval_records
]

print()
print("=" * 78)
print("CONTINUATION COMPLETE")
print("=" * 78)

print(
    "Final optimizer step :",
    end_step,
)

print(
    "Continuation time    :",
    f"{continuation_seconds / 60:.2f} min",
)

print(
    "Initial eval loss    :",
    f"{eval_total[0]:.7f}",
)

print(
    "Latest eval loss     :",
    f"{eval_total[-1]:.7f}",
)

print(
    "Total reduction      :",
    f"{eval_total[0] - eval_total[-1]:.7f}",
)

print(
    "Relative reduction   :",
    f"{100 * (eval_total[0] - eval_total[-1]) / eval_total[0]:.2f}%",
)

print(
    "History              :",
    HISTORY_PATH,
)


# ------------------------------------------------------------
# Plot the complete step 0 -> final trajectory
# ------------------------------------------------------------

plt.figure(figsize=(9, 5))

plt.plot(
    eval_steps,
    eval_total,
    marker="o",
    label="eval-mode total loss",
)

plt.xlabel("Optimizer step")
plt.ylabel("Total loss")
plt.title(
    "STIR-Net same-sample extended overfit"
)

plt.grid(
    True,
    alpha=0.25,
)

plt.legend()
plt.show()


# ------------------------------------------------------------
# Print the loss components at the latest step
# ------------------------------------------------------------

latest = eval_records[-1]["eval_loss"]

print()
print(
    f"Evaluation loss components at step "
    f"{end_step}"
)
print("-" * 48)

for name, value in sorted(
    latest.items()
):
    print(
        f"{name:24s}: "
        f"{float(value):.7f}"
    )